<a id="top"></a>
# House Sales: MLflow tracking e Model Registry (esercitazione – modulo 4)

Questa esercitazione ti guida nell'uso di **MLflow per il tracking e la gestione dei modelli**.

**Obiettivi**:
- Tracciare esperimenti con `autolog`
- Confrontare modelli nella MLflow UI
- Registrare il modello migliore nel Model Registry
- Assegnare alias `production`
- Verificare il funzionamento della webapp

**Prerequisito**: Esercitazione E3 completata (pipeline già definite).

<a id="indice"></a>
## Indice

- [Step 0: Import e setup iniziale](#step0)
- [Step 1: Configurazione MLflow e primo log](#step1)
- [Step 2: Log del secondo modello e confronto nella UI](#step2)
- [Step 3: Registrazione del modello migliore](#step3)
- [Step 4: Assegnazione alias Production](#step4)
- [Step 5: Test della webapp](#step5)
- [Extra (opzionale): Logging avanzato](#extra)


<a id="step0"></a>
## Step 0: Setup iniziale

**Import necessari**:
```python
import mlflow
import mlflow.sklearn
from sklearn.metrics import mean_squared_error, r2_score
# + i tuoi import da E3
```

**Task**:
1. Copia da E3 il codice per caricare dati e creare `X_train`, `X_test`, `y_train`, `y_test`
2. Ricrea le due pipeline: `lr_pipeline` e `rf_pipeline`

**Hint**: Puoi letteralmente copiare le celle da E3!

In [4]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import numpy as np


import pandas as pd
from sklearn.datasets import fetch_openml

# Scarica il dataset House Sales da OpenML
raw = fetch_openml(data_id=42092, as_frame=True)  # House sales
house = raw.frame.copy()

print("Dimensioni del dataset (righe, colonne):", house.shape)
house.head()

#PARTE 2
from sklearn import set_config
from sklearn.model_selection import train_test_split
set_config(transform_output="pandas")


# rimuovere le colonne che non utilizzerò nel modello
colonne_da_rimuovere = [
    "date",
    "zipcode",
    "lat",
    "long",
    "yr_built",
    "yr_renovated"
]

# Separazione tra feature (X) e target (y)
X = house.drop(columns=["price", *colonne_da_rimuovere])
y = house["price"]

# Suddivisione: 70% training, 30% test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

print("Dimensioni di X_train:", X_train.shape)
print("Dimensioni di X_test:", X_test.shape)

#queste sono vuote, non ancora y fitted
print("Dimensioni di y_train:", y_train.shape)
print("Dimensioni di y_test:", y_test.shape)


#PARTE 3
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

# Verifica che tutte le feature siano numeriche
colonne_non_numeriche = X_train.select_dtypes(exclude="number").columns.tolist()
print("Colonne non numeriche:", colonne_non_numeriche)

# Pipeline di preprocessing
preprocessing = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

# Adatta sul training set e trasforma entrambi i set
X_train_preprocessed = preprocessing.fit_transform(X_train)
X_test_preprocessed = preprocessing.transform(X_test)

print("Dimensioni X_train preprocessato:", X_train_preprocessed.shape)
print("Dimensioni X_test preprocessato:", X_test_preprocessed.shape)



#PARTE 4 - MODELLO REGRESSIONE LINEARE
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LinearRegression

modello = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
    ("regression", LinearRegression())
])

# Addestramento
modello.fit(X_train, y_train)

# Previsioni sul test set
y_pred = modello.predict(X_test)

print(y_pred[:10])

#MODELLO RANDOM FOREST

from sklearn.ensemble import RandomForestRegressor

modello2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
    ("regression", RandomForestRegressor(n_estimators=100, random_state=42))
])

# Addestramento
modello2.fit(X_train, y_train)

# Previsioni sul test set
y_pred = modello2.predict(X_test)

print(y_pred[:10])

Dimensioni del dataset (righe, colonne): (21613, 20)
Dimensioni di X_train: (15129, 13)
Dimensioni di X_test: (6484, 13)
Dimensioni di y_train: (15129,)
Dimensioni di y_test: (6484,)
Colonne non numeriche: []
Dimensioni X_train preprocessato: (15129, 13)
Dimensioni X_test preprocessato: (6484, 13)
[ 553966.33266221  752322.13729963 1251867.86602962 1519499.43796542
  728043.44270647  340115.03265771  767415.64039654  499176.76316144
  445794.67807932  323528.86690404]
[ 464398.11  872701.5   982139.   1389269.9   765904.39  331008.2
  523012.5   617722.02  400245.08  527623.54]


<a id="step1"></a>
## Step 1: Primo modello con MLflow

**Configurazione MLflow**:
```python
import os
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")  # necessario con MLflow 3.x (backend su file)

mlflow.set_tracking_uri("file:../mlruns")  # stesso store di bank!
mlflow.set_experiment("house_sales_regression")
mlflow.sklearn.autolog()
```

**Task**: Traccia il primo modello (LinearRegression)
- Usa `mlflow.start_run(run_name="linear_regression")`
- Fai `.fit()` della pipeline
- Calcola RMSE e R² sul test
- Logga con `mlflow.log_metric()`

**Verifica**: dalla root del progetto avvia la UI con
```bash
MLFLOW_ALLOW_FILE_STORE=true uv run mlflow ui --backend-store-uri file:mlruns
```
e controlla l'esperimento su http://127.0.0.1:5000.

💡 Su macOS la porta 5000 può essere occupata (AirPlay Receiver): in quel caso aggiungi `--port 5001`.

In [11]:
# ========================================
# SCRIVI QUI IL TUO CODICE PER LO STEP 1
# ========================================

# Configurazione MLflow
import os

import mlflow

# Opt-in necessario nelle versioni recenti per usare il backend file legacy.
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")

mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("house_sales_regression")

# Autolog per scikit-learn (traccia parametri e metriche automaticamente)
mlflow.sklearn.autolog(
    log_input_examples=True,
    log_model_signatures=True,
    log_models=False
)

print("MLflow configurato:")
print(f"  Tracking URI: {mlflow.get_tracking_uri()}")
print(f"  Experiment: house_sales_regression")
print(f"  Autolog attivo per scikit-learn")



MLflow configurato:
  Tracking URI: file:../mlruns
  Experiment: house_sales_regression
  Autolog attivo per scikit-learn


In [14]:
with mlflow.start_run(run_name="linear_regression"):
    modello.fit(X_train, y_train)
    y_pred_lr = modello.predict(X_test)

    test_rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
    test_r2_lr = r2_score(y_test, y_pred_lr)

    mlflow.log_metric("test_rmse", test_rmse_lr)
    mlflow.log_metric("test_r2", test_r2_lr)

    mlflow.sklearn.log_model(
        sk_model=modello,
        name="model",
        input_example=X_train.head(3),
        serialization_format="cloudpickle"
    )

    print(f"Test RMSE: {test_rmse_lr:.2f}")
    print(f"Test R²: {test_r2_lr:.4f}")

2026/10/08 12:39:03 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\a.ferri\OneDrive - UBroker SpA\Desktop\Ale\corso di formazione\CORSO_IA_DATA_ANALYSIS\ai_course_data_analysis\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/10/08 12:39:03 WARNING mlflow.utils.autologging_utils

Test RMSE: 237950.03
Test R²: 0.6078


<a id="step2"></a>
## Step 2: Secondo modello e confronto

**Task**: Traccia il Random Forest seguendo lo stesso pattern dello Step 1
- Usa `run_name="random_forest"`
- Fit di `rf_pipeline`
- Calcola e logga le stesse metriche

**Confronto nella UI**:
- Aggiorna la pagina (F5)
- Seleziona entrambe le run (checkbox)
- Clicca "Compare"
- Quale modello ha RMSE più basso?

In [15]:
# ========================================
# SCRIVI QUI IL TUO CODICE PER LO STEP 2
# ========================================

# Secondo run: Random Forest
with mlflow.start_run(run_name="random_forest"):
    modello2.fit(X_train, y_train)
    y_pred_rf = modello2.predict(X_test)

    test_rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
    test_r2_rf = r2_score(y_test, y_pred_rf)

    mlflow.log_metric("test_rmse", test_rmse_rf)
    mlflow.log_metric("test_r2", test_r2_rf)

    mlflow.sklearn.log_model(
        sk_model=modello2,
        name="model",
        input_example=X_train.head(3),
        serialization_format="cloudpickle"
    )

    print(f"Test RMSE: {test_rmse_rf:.2f}")
    print(f"Test R²: {test_r2_rf:.4f}")

2026/10/08 12:39:35 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\a.ferri\OneDrive - UBroker SpA\Desktop\Ale\corso di formazione\CORSO_IA_DATA_ANALYSIS\ai_course_data_analysis\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/10/08 12:39:43 WARNING mlflow.utils.autologging_utils

Test RMSE: 211278.31
Test R²: 0.6908


<a id="step3"></a>
## Step 3: Registrazione nel Model Registry

**Dalla MLflow UI**:
1. Identifica il modello migliore (ordina per `test_rmse`)
2. Clicca sulla run → sezione "Logged models" → clicca sul modello `model`
3. Clicca "Register model"
4. **Nome**: `house_sales_rf_prod` (esatto!)
5. Conferma

💡 In MLflow 3 il modello non compare più nella scheda "Artifacts" della run, ma nella sezione "Logged models".

**Verifica**: Menu "Models" → dovresti vedere `house_sales_rf_prod`

⚠️ Il nome deve essere esatto, la webapp lo cerca così!

In [ ]:
# ========================================
# STEP 3: DA COMPLETARE NELLA MLflow UI
# ========================================

# Non serve codice qui!
# Segui le istruzioni nella cella markdown sopra per:
# 1. Identificare il modello migliore
# 2. Registrarlo come 'house_sales_rf_prod'

<a id="step4"></a>
## Step 4: Alias Production

**Dalla MLflow UI**:
1. Menu "Models" → `house_sales_rf_prod`
2. Clicca su Version 1 → accanto ad "Aliases" clicca il pulsante per aggiungere un alias
3. Scrivi `production`
4. Conferma con "Save aliases"

**Verifica**: Dovresti vedere il tag "production" accanto alla versione.

💡 La webapp cerca `models:/house_sales_rf_prod@production`

In [ ]:
# ========================================
# STEP 4: DA COMPLETARE NELLA MLflow UI
# ========================================

# Non serve codice qui!
# Segui le istruzioni nella cella markdown sopra per:
# 1. Assegnare l'alias 'production' alla Version 1

<a id="step5"></a>
## Step 5: Test webapp

**Avvia la webapp**:
```bash
cd Webapp
uv run streamlit run app_house.py
```

**Verifica**:
- Si apre `http://localhost:8501`
- Nella sidebar: gli slider con le caratteristiche dell'abitazione
- Modifica gli slider → le predizioni cambiano
- Prova "Stima il prezzo di vendita"

**Troubleshooting**:
- Errore "Model not found" → controlla nome e alias
- Errore tracking URI → verifica `../mlruns`

✅ Se funziona, hai completato l'esercitazione!

In [ ]:
# ========================================
# STEP 5: TEST DA TERMINALE
# ========================================

# Esegui dal terminale:
# cd Webapp
# uv run streamlit run app_house.py

# Poi verifica nel browser che tutto funzioni!